In [1]:
import numpy as np

def simulasi_bengkel(num_mobil=20, mean_iat=10, mean_reg=2,
                     mean_repair=25, std_repair=5, mean_pay=3, seed=42):
    np.random.seed(seed)

    # 1. Inisialisasi
    s = {"clock": 0.0, "montir_bebas": 2, "resepsionis": "IDLE"}
    antrian_reg, antrian_tunggu, antrian_bayar = [], [], []
    event_calendar = []
    waktu_masuk_tunggu = {}
    total_tunggu, maks_tunggu, selesai = 0.0, 0.0, 0

    def jadwal(waktu, tipe, mobil):
        event_calendar.append([waktu, tipe, mobil])
        event_calendar.sort()

    def layani_resepsionis():
        # kasir (pembayaran) diprioritaskan dibanding pendaftaran
        if s["resepsionis"] != "IDLE":
            return
        if antrian_bayar:
            m = antrian_bayar.pop(0)
            s["resepsionis"] = "BUSY"
            jadwal(s["clock"] + np.random.exponential(mean_pay), "PAY_END", m)
        elif antrian_reg:
            m = antrian_reg.pop(0)
            s["resepsionis"] = "BUSY"
            jadwal(s["clock"] + np.random.exponential(mean_reg), "REG_END", m)

    def mulai_perbaikan(m):
        s["montir_bebas"] -= 1
        durasi = max(5, np.random.normal(mean_repair, std_repair))
        jadwal(s["clock"] + durasi, "REPAIR_END", m)

    # Jadwalkan semua kedatangan
    t = 0.0
    jadwal(t, "ARRIVAL", 1)
    for i in range(2, num_mobil + 1):
        t += np.random.exponential(mean_iat)
        jadwal(t, "ARRIVAL", i)

    print(f"{'Clock':<9}{'Event':<12}{'Mobil':<7}{'Montir bebas':<14}{'Q-Reg':<7}{'Q-Tunggu':<9}{'Q-Bayar'}")
    print("-" * 66)

    # 2. Main loop
    while event_calendar:
        s["clock"], tipe, m = event_calendar.pop(0)

        if tipe == "ARRIVAL":
            antrian_reg.append(m)
            layani_resepsionis()

        elif tipe == "REG_END":
            s["resepsionis"] = "IDLE"
            if s["montir_bebas"] > 0:          # DECISION: montir bebas?
                mulai_perbaikan(m)
            else:
                antrian_tunggu.append(m)        # menunggu di ruang tunggu
                waktu_masuk_tunggu[m] = s["clock"]
            layani_resepsionis()

        elif tipe == "REPAIR_END":
            s["montir_bebas"] += 1
            antrian_bayar.append(m)
            if antrian_tunggu:                  # montir langsung ambil mobil berikutnya
                nxt = antrian_tunggu.pop(0)
                w = s["clock"] - waktu_masuk_tunggu[nxt]
                total_tunggu += w
                maks_tunggu = max(maks_tunggu, w)
                mulai_perbaikan(nxt)
            layani_resepsionis()

        elif tipe == "PAY_END":
            s["resepsionis"] = "IDLE"
            selesai += 1                        # mobil keluar bengkel
            layani_resepsionis()

        print(f"{s['clock']:<9.2f}{tipe:<12}{m:<7}{s['montir_bebas']:<14}"
              f"{len(antrian_reg):<7}{len(antrian_tunggu):<9}{len(antrian_bayar)}")

    # 3. Statistik akhir
    print("\n=== HASIL SIMULASI ===")
    print(f"Total mobil selesai      : {selesai}")
    print(f"Rata-rata tunggu montir  : {total_tunggu / selesai:.2f} menit")
    print(f"Tunggu montir maksimum   : {maks_tunggu:.2f} menit")
    print(f"Total waktu simulasi     : {s['clock']:.2f} menit")

simulasi_bengkel()

Clock    Event       Mobil  Montir bebas  Q-Reg  Q-Tunggu Q-Bayar
------------------------------------------------------------------
0.00     ARRIVAL     1      2             0      0        0
0.69     REG_END     1      1             0      0        0
4.69     ARRIVAL     2      1             0      0        0
5.38     REG_END     2      0             0      0        0
20.62    REPAIR_END  1      1             0      0        0
21.99    PAY_END     1      1             0      0        0
31.95    REPAIR_END  2      2             0      0        0
33.78    PAY_END     2      2             0      0        0
34.79    ARRIVAL     3      2             0      0        0
37.87    REG_END     3      1             0      0        0
47.96    ARRIVAL     4      1             0      0        0
49.76    REG_END     4      0             0      0        0
57.09    ARRIVAL     5      0             0      0        0
57.19    REG_END     5      0             0      1        0
58.79    ARRIVAL     6     